In [1]:
import asyncio
from collections import defaultdict
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score


In [2]:
import sys
from pathlib import Path

# Asumiendo que notebooks/ está dentro de la raíz del proyecto
root_path = Path().resolve().parent  # sube un nivel
sys.path.append(str(root_path))

In [3]:
from agents.classifier_agent import ClassifierAgent
from agents.aggregator_agent import AggregatorAgent
from data.dataset_registry import DatasetRegistry
from data.loaders.sklearn_loader import SklearnLoader
import pandas as pd

In [4]:
import asyncio
import numpy as np
import torch
import pandas as pd

from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier

from data.dataset_registry import DatasetRegistry
from data.loaders.sklearn_loader import SklearnLoader

from agents.classifier_agent import ClassifierAgent
from agents.aggregator_agent import AggregatorAgent

from explainers.shap_explainer import ShapExplainer

from models.sklearn_model import SklearnModel
from models.torch_model import TorchModel
from models.random_forest import RandomForest as AdvancedRF

from visualization.visualization import (
    plot_metrics_over_time,
    plot_explanation_similarity,
    plot_explanation_divergence,
    plot_agents_dashboard
)

from visualization.logs import log

import torch.nn as nn


# -------------------------------------------------------
# MLP deliberadamente malo (underfitting severo)
# -------------------------------------------------------
class BadIrisMLP(nn.Module):
    """Red mínima, sin capacidad real de aprender."""
    def __init__(self, input_dim=4, num_classes=3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 2),   # cuello de botella extremo
            nn.ReLU(),
            nn.Linear(2, num_classes)
        )

    def forward(self, x):
        return self.net(x)


async def run_test():
    print("\n==================== TEST MULTI-AGENT (MALOS MODELOS) ====================")

    # ------------------- Dataset -------------------
    registry = DatasetRegistry()
    dataset_id = "iris"
    registry.register(dataset_id, SklearnLoader(load_iris))

    X, y, meta = registry.load(dataset_id)
    print(f"[Dataset] {dataset_id} cargado → X={X.shape}, y={y.shape}")

    instance = X[:1]

    # -------------------------------------------------------
    # Modelos con hiperparámetros malos a propósito
    # -------------------------------------------------------

    # RF base: 1 árbol, sin aleatorización → muy propenso a overfitting puntual
    rf_base = SklearnModel(
        RandomForestClassifier(
            n_estimators=1,      # un solo árbol
            max_depth=1,         # árbol tocón
            max_features=1,      # solo 1 feature por split
            random_state=0
        )
    )

    # RF avanzado: excesivamente limitado
    rf_adv = AdvancedRF(
        n_estimators=2,
        max_depth=1,             # incapaz de capturar relaciones no lineales
        random_state=1
    )

    # Torch: lr altísimo + pocas épocas → entrenamiento inestable
    torch_model = TorchModel(
        nn_model=BadIrisMLP(input_dim=X.shape[1]),
        lr=0.9,                  # lr agresivo → divergencia del gradiente
        epochs=2,                # casi sin entrenamiento
        batch_size=128           # batch grande en dataset pequeño
    )

    # ------------------- Clasificadores -------------------
    classifiers = {
        "rf_base_bad": ClassifierAgent(
            agent_id="rf_base_bad",
            model=rf_base,
            explainers=[ShapExplainer()],
            dataset_id=dataset_id,
            registry=registry
        ),
        "rf_adv_bad": ClassifierAgent(
            agent_id="rf_adv_bad",
            model=rf_adv,
            explainers=[ShapExplainer()],
            dataset_id=dataset_id,
            registry=registry
        ),
        "nn_torch_bad": ClassifierAgent(
            agent_id="nn_torch_bad",
            model=torch_model,
            explainers=[ShapExplainer()],
            dataset_id=dataset_id,
            registry=registry
        )
    }

    classifier_ids = list(classifiers.keys())

    # -------------------------------------------------------
    # Aggregator: más iteraciones para ver evolución
    # -------------------------------------------------------
    aggregator = AggregatorAgent(
        classifier_ids=classifier_ids,
        max_iterations=10,   # ← suficientes para ver tendencias
        alpha=0.4,
        beta=0.15,
        gamma=0.25
    )

    queues = {cid: agent.inbox for cid, agent in classifiers.items()}
    queues["aggregator"] = aggregator.inbox

    # ------------------- Setup -------------------
    print("\n[Setup] Inicializando agentes...\n")
    await asyncio.gather(*(agent.setup() for agent in classifiers.values()))
    log("Inicializando AggregatorAgent")

    # ------------------- Tasks -------------------
    tasks = [
        asyncio.create_task(agent.run(queues))
        for agent in classifiers.values()
    ]
    tasks.append(
        asyncio.create_task(aggregator.run(queues, instance))
    )

    await asyncio.gather(*tasks)

    print("\n==================== TEST FINALIZADO ✅ ====================\n")

    # =========================================================
    # VISUALIZACIÓN
    # =========================================================
    print("\n[Visualización] Métricas por agente\n")

    for agent in classifiers.values():
        plot_metrics_over_time(agent.metrics_history, metric_name="accuracy")
        plot_metrics_over_time(agent.metrics_history, metric_name="f1")
        plot_explanation_similarity(agent)

    print("\n[Visualización] Divergencia global de explicaciones\n")
    plot_explanation_divergence(classifiers)

    print("\n[Visualización] Dashboard global\n")
    plot_agents_dashboard(classifiers)

    # =========================================================
    # DataFrame de resultados por iteración
    # =========================================================
    rows = []

    for entry in aggregator.global_history:
        iteration = entry["iteration"]
        evaluation = entry["evaluation"]
        exp_detail = evaluation["components"]["exp_detail"]

        for idx, agent_id in enumerate(classifier_ids):
            row = {
                "agent":        agent_id,
                "iteration":    iteration,
                "accuracy":     entry["responses"][idx]["metrics"]["accuracy"],
                "f1":           entry["responses"][idx]["metrics"]["f1"],
                "precision":    entry["responses"][idx]["metrics"]["precision"],
                "recall":       entry["responses"][idx]["metrics"]["recall"],
                "exp_quality":  exp_detail["quality"][idx],
                "exp_consensus":exp_detail["consensus"][idx],
                "exp_stability":exp_detail["stability"][idx],
            }
            rows.append(row)

    df = pd.DataFrame(rows)
    df_sorted = df.sort_values(["agent", "iteration"])

    # Resumen rápido para inspección
    print("\n[Resumen] Medias por agente:")
    print(df_sorted.groupby("agent")[["accuracy","f1","exp_quality","exp_stability"]].mean().round(3))

    return df_sorted

In [5]:
await run_test()



==================== TEST MULTI-AGENT (MALOS MODELOS) ====================
[Dataset] iris cargado → X=(150, 4), y=(150,)

[Setup] Inicializando agentes...

[rf_base_bad] Setup iniciado
[rf_base_bad] 17:04:22 | Setup iniciado (train + eval inicial)
[rf_base_bad] 17:04:22 | Entrenando modelo (iter=0)
[rf_base_bad] 17:04:22 | Evaluación completada | acc=0.600
[rf_base_bad] 17:04:22 | Setup completado
[rf_adv_bad] Setup iniciado
[rf_adv_bad] 17:04:22 | Setup iniciado (train + eval inicial)
[rf_adv_bad] 17:04:22 | Entrenando modelo (iter=0)
[rf_adv_bad] 17:04:22 | Evaluación completada | acc=0.700
[rf_adv_bad] 17:04:22 | Setup completado
[nn_torch_bad] Setup iniciado
[nn_torch_bad] 17:04:22 | Setup iniciado (train + eval inicial)
[nn_torch_bad] 17:04:22 | Entrenando modelo (iter=0)
[nn_torch_bad] 17:04:22 | Evaluación completada | acc=0.333
[nn_torch_bad] 17:04:22 | Setup completado


c:\Users\saulr\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\saulr\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\saulr\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(ave

CancelledError: 